# TGLF physics knobs on the reference KBM: right eigenfunction, wrong $\gamma$ (Fusion_PhD-qp5.47)

One mode: SPR-045 `kbm_free` `sample_0818`, the reference KBM ($k_y\rho = 0.0198$; GS2 $\gamma = 1.371$, $\omega = +0.176$ $c_s/a$).
At WIDTH 0.4 and NBASIS $\ge 8$ TGLF converges to GS2's eigenfunction extent (`results/spr045_single_kbm` v2) but to about twice GS2's
$\gamma$. Which TGLF physics knob closes the gap without changing the mode?

**Runs** (TGLF only; leaf `TGLF/Runs/single_mode_spr045/SPR-045/knobs_sample_0818`, registry family `tglf_single_kbm_knobs`): copies
of the one deck. Baseline = qp5.41 v2 / qp5.44: WIDTH 0.4, `FIND_WIDTH` F, SAT_RULE 2, FILTER $-1.5$, NXGRID 32, IBRANCH $-1$, MHD rule
off, $A_\parallel$ and $B_\parallel$ on, NMODES 4, UNITS CGYRO, NBASIS 24 and 16. One knob changed per run, each a per-sample variable of
the leaf's `pyroscan.nc`. Baseline also at WIDTH 0.3, 0.5, 0.6.

**What each knob is** (TGLF source, `$HOME/gacode/tglf/src`):
- THETA_TRAPPED: bounce-length boundary of the trapped fraction, $f_t = \sqrt{1 - B_{\min}/B(y_b)}$ with
  $y_b = \min(L_y, \mathrm{THETA\_TRAPPED}\cdot\mathrm{WIDTH}\,L_y/\sqrt2)$ (`tglf_geometry.f90:740-760`); saturates at $\sqrt2/\mathrm{WIDTH} = 3.5355$.
- XNU_FACTOR: multiplies the trapped-electron collision coefficient (`tglf_eigensolver.f90:455-480`).
- PARK: the Landau-closure parallel wavenumber $k_{\parallel 0} = \mathrm{PARK}/(R\,q\,\mathrm{WIDTH})$ (`tglf_eigensolver.f90:123`).
- GHAT: the magnetic-drift frequency of the non-adiabatic moment equations, $\omega_{d1} = -\mathrm{GHAT}\,k_y/R$ (`:1039`).
- GCHAT: the drift coupling in the field equations, $\omega_{cd} = -\mathrm{GCHAT}\,k_y/R$ (`:125`, used in the $\phi$ and $\psi$ rows, e.g. `:1180`, `:1264`).
- LINSKER_FACTOR: finite-Larmor-radius gradient (Linsker) terms, $0.5\times$ the factor (`:245`, `:813`); off by default.
- GRADB_FACTOR: mirror-force ($\nabla B$) terms, $\propto k_{\parallel 1}\,\mathrm{GRADB\_FACTOR}$ (`:1052`); off by default.
- DAMP_PSI, DAMP_SIG: enter only $\beta_{e,\psi}$, $\beta_{e,\sigma}$ at NBASIS 2 (`:211-240`; the `damp_psi` line itself is commented out),
  so at NBASIS 16/24 they are expected to do nothing; run at 0, 1, 4 as a null check.
- SAT_RULE 1 vs 2: in a linear run only its presets act (`tglf_startup.f90:45-63`, `USE_PRESETS` hard-coded): SAT2 = XNU_MODEL 3,
  WDIA_TRAPPED 1; SAT1 = XNU_MODEL 2, WDIA_TRAPPED 0. UNITS CGYRO pinned in both.

**Measures** (dominant mode = argmax $\gamma$ over `mode`; IBRANCH $-1$ so it is mode 0, asserted): $\gamma/\gamma_{\rm GS2}$,
$\omega/\omega_{\rm GS2}$, the pyro 95 % `Extent` of $|\phi|^2$ and $|A_\parallel|^2$ over GS2's, and the shape overlap
$$O_f = \frac{\left|\int f_{\rm TGLF}^*\, f_{\rm GS2}\, d\theta\right|}{\lVert f_{\rm TGLF}\rVert\,\lVert f_{\rm GS2}\rVert},\qquad f \in \{\phi, A_\parallel\},$$
with TGLF interpolated onto GS2's $\theta$ grid (zero outside TGLF's range). $O_f = 1$ means the same shape, whatever the phase and
amplitude. GS2 $\gamma$, $\omega$ and extents come from the `kbm_free` cube; its complex eigenfunction from the `complex_sample_0818` re-save.

**Stated defaults (the user's to change):** "within $x$" = $|\gamma/\gamma_{\rm GS2} - 1| \le x$; a run "keeps the mode" if
$\omega > 0$, $O_\phi \ge 0.95$, $O_{A_\parallel} \ge 0.9$ and both extent ratios within 0.8–1.25; a dominant mode with $\omega \le 0$ or
$O_\phi$ below 0.8 against the baseline TGLF mode is marked as another branch (dotted in the eigenfunction figures).

In [ ]:
import os
from pathlib import Path
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.ticker import StrMethodFormatter
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
W = plt.rcParams["figure.figsize"][0]  # paper \textwidth; figures are made at printed size
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"])).expanduser()     # the WORK tree
plots_dir = ROOT / "Plots" / "TGLF_GFTM_KBM_PAPER" / "single_kbm_tglf_knobs"
plots_dir.mkdir(parents=True, exist_ok=True)

# Fusion_PhD-whvp: numpy 2.1 on Python 3.14 in a Jupyter kernel overwrote operands of large array subtractions. Fail loudly if this runtime does.
def subtraction_probe(size=40001):
    left = np.ones(size)
    difference = left - 0.25 * np.ones(size)
    return left[0], difference[0]
assert subtraction_probe() == (1.0, 0.75), "this runtime corrupts array arithmetic (Fusion_PhD-whvp): run as plain python"

case, sample = "SPR-045", "sample_0818"
gs2_cube = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
gs2_complex = data_root / "GS2/Runs/single_mode_spr045" / case / f"complex_{sample}"
tglf_leaf = data_root / "TGLF/Runs/single_mode_spr045" / case / f"knobs_{sample}"
gftm_leaf = data_root / "GFTM/Runs/single_mode_spr045" / case / f"galF_v2_{sample}"    # qp5.41 stock GFTM, FILTER 0.5
fraction = 0.95                                  # pyro Extent: theta range holding 95 % of |field|^2
fields = ["phi", "apar"]
field_labels = {"phi": r"$\phi$", "apar": r"$A_\parallel$"}
width, nbasis, nbasis_check = 0.4, 24, 16
defaults = {"THETA_TRAPPED": 0.7, "XNU_FACTOR": 1.0, "PARK": 1.0, "GHAT": 1.0, "GCHAT": 1.0, "DAMP_PSI": 0.0, "DAMP_SIG": 0.0,
            "LINSKER_FACTOR": 0.0, "GRADB_FACTOR": 0.0, "SAT_RULE": 2}
log_knobs = {"THETA_TRAPPED": [0.05, 0.1, 0.3, 1, 3], "XNU_FACTOR": [0.01, 0.1, 1, 10, 100], "PARK": [0.25, 0.5, 1, 2, 4]}   # knob: x ticks
keeps_mode = dict(overlap_phi=0.95, overlap_apar=0.9, extent_low=0.8, extent_high=1.25)
branch_overlap = 0.8

## Data: GS2 reference, the TGLF knob leaf, GFTM v2

In [ ]:
gs2_scan = PyroHypercube(pyroscan_json=gs2_cube / "pyroscan.json", load_base_pyro=True)
gs2_scan.load_gk_output(netcdf_file=gs2_cube / "pyroscan.nc")
sample_index = list(gs2_scan.gk_output.data.sample_name.values.astype(str)).index(sample)
gs2_output = gs2_scan.gk_output.data.isel(sample=sample_index).pint.dequantify()
gamma_gs2, omega_gs2 = float(gs2_output.growth_rate), float(gs2_output.mode_frequency)
extent_gs2 = {field: float(Extent._compute(gs2_output.eigenfunctions_squared.sel(field=field).isel(kx=0).dropna("theta"), fraction)[0])
              for field in fields}

# the complex re-save has no pyroscan.json; it pairs with the one GS2 run, whose deck exists only on the scratch tree (as single_kbm_gs2_eigenfunction)
gs2_run_root = Path(os.environ["GYRO_DATA_OUTPUT"]) / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
complex_scan = PyroHypercube.from_directory(gs2_run_root, pattern=sample, params=["ky"], gk_code="GS2", file_name="input.in")
complex_scan.load_gk_output(netcdf_file=gs2_complex / "pyroscan.nc")
gs2_mode = complex_scan.gk_output.data.isel(sample=0).pint.dequantify().eigenfunctions.squeeze()
assert np.iscomplexobj(gs2_mode) and set(gs2_mode.dims) == {"field", "theta"}
print(f"GS2 {sample}: gamma {gamma_gs2:.4f}, omega {omega_gs2:+.4f}, extent", {f: round(v, 3) for f, v in extent_gs2.items()})

tglf_scan = PyroHypercube(pyroscan_json=tglf_leaf / "pyroscan.json", load_base_pyro=True)
tglf_scan.load_gk_output(netcdf_file=tglf_leaf / "pyroscan.nc")
Extent(tglf_scan.gk_output, fraction=fraction)
tglf = tglf_scan.gk_output.data.pint.dequantify().squeeze("ky", drop=True)   # one ky per run

gftm_scan = PyroHypercube(pyroscan_json=gftm_leaf / "pyroscan.json", load_base_pyro=True)
gftm_scan.load_gk_output(netcdf_file=gftm_leaf / "pyroscan.nc")
gftm = gftm_scan.gk_output.data.pint.dequantify().squeeze("ky", drop=True)
print("TGLF runs", tglf.sizes["sample"], "| GFTM v2 runs", gftm.sizes["sample"], "| TGLF modes", tglf.sizes["mode"])

## Per-run measures

Normalisation for every eigenfunction plot: the phase is fixed so $\phi$ is real and positive at its $|\phi|$ peak (pyro's convention;
$A_\parallel$ is odd with two equal peaks, so its own peak cannot fix a sign), then each field is divided by its own peak $|f|$. The
overlap $O_f$ does not depend on either choice. Besides the dominant mode, each run's **most GS2-like unstable mode** (largest $O_\phi$ of
the 4) is tracked, to follow the KBM branch through a change of dominant mode.

In [ ]:
def normalised(eigenfunctions):
    eigenfunctions = eigenfunctions.squeeze()
    phi = eigenfunctions.sel(field="phi")
    peak = phi.isel(theta=int(np.abs(phi).argmax("theta")))
    rotated = eigenfunctions / (peak / np.abs(peak))
    return rotated / np.abs(rotated).max("theta")


def overlap(field, reference):
    # |<f, f_ref>| / (||f|| ||f_ref||) on the reference's theta grid; f is zero outside its own theta range
    on_grid = (np.interp(reference.theta, field.theta, field.real, left=0, right=0)
               + 1j * np.interp(reference.theta, field.theta, field.imag, left=0, right=0))
    inner = lambda a, b: np.trapezoid(np.conj(a) * b, reference.theta)
    return float(np.abs(inner(on_grid, reference.values)) / np.sqrt(np.abs(inner(on_grid, on_grid)) * np.abs(inner(reference.values, reference.values))))


gs2_shapes = normalised(gs2_mode)
dominant_index = tglf.growth_rate.fillna(-np.inf).argmax("mode")
assert (dominant_index == 0).all(), "IBRANCH -1 sorts modes by growth rate, so mode 0 should be dominant"
dominant = tglf.isel(mode=dominant_index)

knobs = pd.DataFrame({key: tglf[key].values.astype(float) for key in ["WIDTH", "NBASIS_MAX", *defaults]})
knobs["WIDTH"] = knobs.WIDTH.round(4)
table = knobs.assign(
    gamma_ratio=dominant.growth_rate.values / gamma_gs2, omega_ratio=dominant.mode_frequency.values / omega_gs2,
    **{f"extent_ratio_{field}": dominant.extent.sel(field=field).values / extent_gs2[field] for field in fields},
    **{f"overlap_{field}": [overlap(dominant.eigenfunctions.sel(field=field).isel(sample=run).squeeze(), gs2_shapes.sel(field=field))
                            for run in range(tglf.sizes["sample"])] for field in fields})
is_baseline = (table[list(defaults)] == pd.Series(defaults)).all(axis=1)
baseline_run = {nb: int(table[is_baseline & (table.WIDTH == width) & (table.NBASIS_MAX == nb)].index[0]) for nb in (nbasis, nbasis_check)}
baseline_phi = {nb: dominant.eigenfunctions.sel(field="phi").isel(sample=run).squeeze() for nb, run in baseline_run.items()}
table["overlap_phi_vs_baseline"] = [overlap(dominant.eigenfunctions.sel(field="phi").isel(sample=run).squeeze(), baseline_phi[int(nb)])
                                    for run, nb in enumerate(table.NBASIS_MAX)]
# the most GS2-like unstable mode of each run: largest O_phi among the modes with gamma > 0
overlap_all = np.array([[overlap(tglf.eigenfunctions.sel(field="phi").isel(sample=run, mode=mode).squeeze(), gs2_shapes.sel(field="phi"))
                         if float(tglf.growth_rate.isel(sample=run, mode=mode).squeeze()) > 0 else -1.0 for mode in range(tglf.sizes["mode"])]
                        for run in range(tglf.sizes["sample"])])
gs2_like = tglf.isel(mode=xr.DataArray(overlap_all.argmax(axis=1), dims="sample"))
table["gamma_ratio_gs2_like"] = gs2_like.growth_rate.values.squeeze() / gamma_gs2
table["omega_ratio_gs2_like"] = gs2_like.mode_frequency.values.squeeze() / omega_gs2
table["overlap_phi_gs2_like"] = overlap_all.max(axis=1)
table["other_branch"] = (table.omega_ratio <= 0) | (table.overlap_phi_vs_baseline < branch_overlap) | ~(table.gamma_ratio > 0)
table["keeps_mode"] = ((table.omega_ratio > 0) & (table.overlap_phi >= keeps_mode["overlap_phi"]) & (table.overlap_apar >= keeps_mode["overlap_apar"])
                       & table.extent_ratio_phi.between(keeps_mode["extent_low"], keeps_mode["extent_high"])
                       & table.extent_ratio_apar.between(keeps_mode["extent_low"], keeps_mode["extent_high"]))


def knob_scan(knob, nb, scan_width=width):
    # runs at this WIDTH and NBASIS where every knob except `knob` is at its default, sorted by the knob value
    others = [key for key in defaults if key != knob]
    return table[(table.WIDTH == scan_width) & (table.NBASIS_MAX == nb) & (table[others] == pd.Series(defaults)[others]).all(axis=1)].sort_values(knob)


measures = ["gamma_ratio", "omega_ratio", "extent_ratio_phi", "extent_ratio_apar", "overlap_phi", "overlap_apar"]
print("baseline (WIDTH 0.4):\n", table.loc[list(baseline_run.values()), ["NBASIS_MAX", *measures]].round(3).to_string(index=False))
print("baseline WIDTH robustness:\n", table[is_baseline]
      .sort_values(["NBASIS_MAX", "WIDTH"])[["WIDTH", "NBASIS_MAX", *measures]].round(3).to_string(index=False))

## 1. Baseline: right eigenfunction, wrong $\gamma$?

TGLF at WIDTH 0.3–0.6, NBASIS 16 and 24 (colour by WIDTH, NBASIS 16 dotted), GFTM stock at WIDTH 0.4, NBASIS 32 and 56 (qp5.41 leaf), GS2
black dashed. Rows: $\phi$, $A_\parallel$; columns: $|f|$, ${\rm Re}\,f$, ${\rm Im}\,f$, each field normalised to its own peak.

In [ ]:
width_colours = dict(zip([0.3, 0.4, 0.5, 0.6], plt.cm.viridis(np.linspace(0, 0.85, 4))))
gftm_dominant = gftm.isel(mode=gftm.growth_rate.fillna(-np.inf).argmax("mode"))
gftm_runs = {nb: int(np.flatnonzero((np.abs(gftm.WIDTH.values - width) < 1e-3) & (gftm.NBASIS_MAX.values == nb))[0]) for nb in (32, 56)}
fig_baseline, axes = plt.subplots(2, 3, figsize=(W, 0.6 * W), sharex=True, sharey="row")
for row, field in enumerate(fields):
    for col, part in enumerate([np.abs, np.real, np.imag]):
        ax = axes[row, col]
        for nb, ls in ((nbasis, "-"), (nbasis_check, ":")):
            for scan_width, colour in width_colours.items():
                run = int(table[is_baseline & (table.WIDTH == scan_width) & (table.NBASIS_MAX == nb)].index[0])
                shape = normalised(dominant.eigenfunctions.isel(sample=run)).sel(field=field)
                ax.plot(shape.theta / np.pi, part(shape), color=colour, ls=ls, marker="", label=f"TGLF W {scan_width:g}" if nb == nbasis else None)
        for nb, colour in ((32, "C1"), (56, "C3")):
            shape = normalised(gftm_dominant.eigenfunctions.isel(sample=gftm_runs[nb])).sel(field=field)
            ax.plot(shape.theta / np.pi, part(shape), color=colour, marker="", label=f"GFTM NBASIS {nb}")
        ax.plot(gs2_shapes.theta / np.pi, part(gs2_shapes.sel(field=field)), color="k", ls="--", marker="", label="GS2")
        ax.set_xlim(-3, 3)
    axes[row, 0].set_ylabel(field_labels[field])
for ax, title in zip(axes[0], ["|f| / peak", "Re f / peak", "Im f / peak"]):
    ax.set_title(title)
for ax in axes[-1]:
    ax.set_xlabel(r"$\theta/\pi$")
axes[0, 0].legend(fontsize="x-small")
gamma_line = ", ".join(f"NB{nb} {table.gamma_ratio[run]:.2f}" for nb, run in baseline_run.items())
fig_baseline.suptitle(rf"{case} {sample}, TGLF W 0.4 $\gamma/\gamma_{{GS2}}$: {gamma_line}; GFTM NB32/56: "
                      + ", ".join(f"{float(gftm_dominant.growth_rate[run]) / gamma_gs2:.2f}" for run in gftm_runs.values()))
plt.show()

## 2. One knob at a time: $\gamma$, $\omega$, extents and overlap

WIDTH 0.4. Dominant mode at NBASIS 24 solid with markers, NBASIS 16 dotted; the most GS2-like unstable mode orange; every unstable mode
of the NBASIS 24 run faint grey (catches branch switching; the $\omega$ axis is cut to the dominant and GS2-like range, so far-off modes
can fall outside it); GS2 = 1 (black dashed); the baseline value grey vertical line.

In [ ]:
gamma_all = tglf.growth_rate.squeeze().transpose("sample", "mode").values / gamma_gs2
omega_all = tglf.mode_frequency.squeeze().transpose("sample", "mode").values / omega_gs2
fig_knobs = {}
for knob in defaults:
    fig, axes = plt.subplots(2, 3, figsize=(W, 0.55 * W), sharex=True)
    for ax, measure in zip(axes.flat, measures):
        for nb, style in ((nbasis, dict(ls="-", marker="o", ms=3)), (nbasis_check, dict(ls=":", marker=""))):
            scan = knob_scan(knob, nb)
            ax.plot(scan[knob], scan[measure], color="C0", label=f"NBASIS {nb}", **style)
        if measure in ("gamma_ratio", "omega_ratio"):
            scan = knob_scan(knob, nbasis)
            ax.plot(scan[knob], scan[f"{measure}_gs2_like"], color="C1", marker="", label="most GS2-like mode")
            shown = pd.concat([scan[measure], scan[f"{measure}_gs2_like"]]).dropna()
            ax.set_ylim(min(shown.min(), 0) - 0.1 * np.ptp(shown) - 0.1, max(shown.max(), 1) + 0.1 * np.ptp(shown) + 0.1)
            values = gamma_all if measure == "gamma_ratio" else omega_all
            for mode in range(tglf.sizes["mode"]):
                unstable = gamma_all[scan.index, mode] > 0
                ax.plot(scan[knob][unstable], values[scan.index, mode][unstable], ls="", marker=".", color="0.6", alpha=0.6, zorder=0)
        ax.axhline(1, color="k", ls="--", marker="")
        ax.axvline(defaults[knob], color="0.7", lw=1, marker="")
        ax.set(ylabel={"gamma_ratio": r"$\gamma/\gamma_{GS2}$", "omega_ratio": r"$\omega/\omega_{GS2}$",
               "extent_ratio_phi": r"$\phi$ extent ratio", "extent_ratio_apar": r"$A_\parallel$ extent ratio",
               "overlap_phi": r"overlap $O_\phi$", "overlap_apar": r"overlap $O_{A_\parallel}$"}[measure])
    for ax in axes[-1]:
        ax.set_xlabel(knob)
        if knob in log_knobs:
            ax.set(xscale="log", xticks=log_knobs[knob])
            ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
            ax.minorticks_off()
    axes[0, 0].legend(fontsize="x-small")
    fig.suptitle(f"{case} {sample}, TGLF WIDTH {width:g}: {knob}")
    fig_knobs[knob] = fig
    plt.show()
    print(knob_scan(knob, nbasis)[[knob, *measures, "gamma_ratio_gs2_like", "omega_ratio_gs2_like", "overlap_phi_gs2_like", "other_branch", "keeps_mode"]]
          .round(3).to_string(index=False))

## 3. How the eigenfunction changes with each knob

Dominant mode at NBASIS 24, WIDTH 0.4, one line per knob value coloured by the value (log colour scale for THETA_TRAPPED, XNU_FACTOR and
PARK), the baseline bold, GS2 black dashed. Phase fixed so $\phi$ is real and positive at its $|\phi|$ peak, then each field divided by its
own peak $|f|$. Dotted: a dominant mode on another branch ($\omega \le 0$, or $O_\phi < 0.8$ against the baseline TGLF mode).

In [ ]:
def eigenfunction_evolution(scan, knob, title):
    norm = (LogNorm if knob in log_knobs else Normalize)(scan[knob].min(), scan[knob].max())
    fig, axes = plt.subplots(2, 3, figsize=(W, 0.6 * W), sharex=True, sharey="row", layout="constrained")
    for run, value, other in zip(scan.index, scan[knob], scan.other_branch):
        shape = normalised(dominant.eigenfunctions.isel(sample=run))
        for row, field in enumerate(fields):
            for col, part in enumerate([np.abs, np.real, np.imag]):
                axes[row, col].plot(shape.theta / np.pi, part(shape.sel(field=field)), color=plt.cm.viridis(norm(value)), marker="",
                                    lw=2.5 if value == defaults[knob] else 1, ls=":" if other else "-")
    for row, field in enumerate(fields):
        for col, part in enumerate([np.abs, np.real, np.imag]):
            axes[row, col].plot(gs2_shapes.theta / np.pi, part(gs2_shapes.sel(field=field)), color="k", ls="--", marker="", label="GS2")
            axes[row, col].set_xlim(-3, 3)
        axes[row, 0].set_ylabel(field_labels[field])
    for ax, column_title in zip(axes[0], ["|f| / peak", "Re f / peak", "Im f / peak"]):
        ax.set_title(column_title)
    for ax in axes[-1]:
        ax.set_xlabel(r"$\theta/\pi$")
    axes[0, 0].legend(fontsize="x-small")
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="viridis"), ax=axes, label=knob)
    switched = scan[scan.other_branch][knob].tolist()
    fig.suptitle(title + (f" (other branch at {knob} {switched})" if switched else ""))
    plt.show()
    return fig


fig_shapes = {knob: eigenfunction_evolution(knob_scan(knob, nbasis), knob, f"{case} {sample}, TGLF W {width:g} NBASIS {nbasis}: {knob}")
              for knob in defaults}

## 4. Summary: which knob brings $\gamma$ to GS2 and keeps the mode?

Each knob's path in ($\gamma/\gamma_{\rm GS2}$, $\omega/\omega_{\rm GS2}$) and ($\gamma/\gamma_{\rm GS2}$, $O_\phi$), dominant mode, NBASIS 24. The
$\omega$ axis is cut at $-1.5$; knobs with no effect sit on the star.

In [ ]:
fig_summary, axes = plt.subplots(1, 2, figsize=(W, 0.45 * W))
for i, knob in enumerate(defaults):
    scan = knob_scan(knob, nbasis)
    axes[0].plot(scan.gamma_ratio, scan.omega_ratio, color=f"C{i}", marker="o", ms=2, label=knob)
    axes[1].plot(scan.gamma_ratio, scan.overlap_phi, color=f"C{i}", marker="o", ms=2)
for ax in axes:
    ax.axvline(1, color="k", ls="--", marker="")
    ax.axvspan(0.8, 1.2, color="0.9", zorder=0)
    ax.plot(*table.loc[baseline_run[nbasis], ["gamma_ratio", "omega_ratio" if ax is axes[0] else "overlap_phi"]], "k*", ms=10)
    ax.set(xscale="log", xlabel=r"$\gamma/\gamma_{GS2}$")
axes[0].axhline(1, color="k", ls="--", marker="")
axes[0].set(ylabel=r"$\omega/\omega_{GS2}$", ylim=(-1.5, 3))   # cut: XNU_FACTOR >= 10 and GHAT 0 reach -15 (tables above)
axes[1].set(ylabel=r"overlap $O_\phi$")
axes[0].legend(fontsize="x-small", ncol=2)
fig_summary.suptitle(r"each knob's path at NBASIS 24 (star: baseline; band: $\gamma$ within 20 %)")
plt.show()

table["omega_within_20pct"] = (table.omega_ratio - 1).abs() <= 0.2
answers = []
for knob in defaults:
    for tolerance in (0.1, 0.2):
        scan = knob_scan(knob, nbasis)
        hits = scan[((scan.gamma_ratio - 1).abs() <= tolerance)]
        answers.append(dict(knob=knob, tolerance=tolerance, values_within=hits[knob].tolist(), keeps_mode=hits[hits.keeps_mode][knob].tolist(),
                            keeps_mode_and_omega=hits[hits.keeps_mode & hits.omega_within_20pct][knob].tolist()))
answers = pd.DataFrame(answers)
print(answers.to_string(index=False))

## 5. The best pair: PARK $\times$ GHAT

NBASIS 24, WIDTH 0.4, all other knobs at default. One line per GHAT against PARK (GHAT 1 is the 1-D PARK scan, PARK 1 the 1-D GHAT scan; the
1-D grids lack PARK 0.8 and 0.9 at GHAT 1 and GHAT 0.7 and 0.8 at PARK 1, so those lines have gaps there). Then one eigenfunction figure per
GHAT row, lines coloured by PARK.

In [ ]:
park_range, ghat_values = (0.75, 1.35), [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
others = [key for key in defaults if key not in ("PARK", "GHAT")]
pair = table[(table.WIDTH == width) & (table.NBASIS_MAX == nbasis) & (table[others] == pd.Series(defaults)[others]).all(axis=1)
             & table.PARK.between(*park_range) & table.GHAT.isin(ghat_values)]
ghat_norm = Normalize(min(ghat_values), max(ghat_values))
pair_measures = {"gamma_ratio": r"$\gamma/\gamma_{GS2}$", "omega_ratio": r"$\omega/\omega_{GS2}$", "overlap_phi": r"overlap $O_\phi$",
                 "extent_ratio_phi": r"$\phi$ extent ratio"}
fig_pair, axes = plt.subplots(2, 2, figsize=(W, 0.6 * W), sharex=True, layout="constrained")
for ghat, rows in pair.groupby("GHAT"):
    rows = rows.sort_values("PARK")
    for ax, measure in zip(axes.flat, pair_measures):
        ax.plot(rows.PARK, rows[measure], color=plt.cm.viridis(ghat_norm(ghat)), ms=3, label=f"GHAT {ghat:g}")
for ax, (measure, label) in zip(axes.flat, pair_measures.items()):
    ax.axhline(1, color="k", ls="--", marker="")
    ax.set_ylabel(label)
for ax in axes[-1]:
    ax.set_xlabel("PARK")
axes[0, 0].legend(fontsize="x-small", ncol=2)
fig_pair.suptitle(f"{case} {sample}, TGLF W {width:g} NBASIS {nbasis}: PARK x GHAT")
plt.show()
for measure in ["gamma_ratio", "omega_ratio", "overlap_phi"]:
    print(measure, "\n", pair.pivot_table(index="GHAT", columns="PARK", values=measure).round(3).to_string())
best_pairs = pair[((pair.gamma_ratio - 1).abs() <= 0.1) & pair.keeps_mode & pair.omega_within_20pct]
print("gamma within 10 %, mode kept, omega within 20 %:\n", best_pairs[["PARK", "GHAT", *measures]].round(3).to_string(index=False))
fig_pair_shapes = {ghat: eigenfunction_evolution(rows.sort_values("PARK"), "PARK", f"{case} {sample}, TGLF W {width:g} NBASIS {nbasis}: PARK at GHAT {ghat:g}")
                   for ghat, rows in pair.groupby("GHAT")}

## Save

In [ ]:
for name, fig in [("baseline_eigenfunctions", fig_baseline), ("summary", fig_summary),
                  *[(f"knob_{knob}", fig) for knob, fig in fig_knobs.items()], *[(f"eigenfunction_vs_{knob}", fig) for knob, fig in fig_shapes.items()],
                  ("park_x_ghat", fig_pair), *[(f"eigenfunction_vs_PARK_at_GHAT_{ghat:g}".replace(".", "p"), fig) for ghat, fig in fig_pair_shapes.items()]]:
    fig.savefig(plots_dir / f"{name}.pdf")
    fig.savefig(plots_dir / f"{name}.png", dpi=200)
table.round(5).to_csv(plots_dir / "knob_runs.csv", index_label="sample_index")
answers.to_csv(plots_dir / "knob_answers.csv", index=False)
best_pairs.round(5).to_csv(plots_dir / "park_x_ghat_best.csv", index_label="sample_index")
print(sorted(path.name for path in plots_dir.iterdir()))